In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "اكتب جملة عربية بسيطة فيها خطأ إملائي شائع"}]
)
print(response.choices[0].message.content)

سألتقي صديقي في الحديقة **الى** مساءً.


In [ ]:
import json

def correct_spelling(text):
    system_prompt = """أنت مدرّس عربي لطيف يساعد طفلًا عمره 7-10 سنوات على تحسين كتابته بالعربية الفصحى.

اكتشفي الأخطاء بالفئات التالية فقط:

١. الإملاء: التاء المربوطة والهاء، الهمزات (أ/إ/ا)، الألف المقصورة والياء (ى/ي)
٣. أل التعريف: أسماء المؤسسات والأنشطة الروتينية (مدرسة، جامعة، مستشفى) عادة تحتاج "ال" عند الحديث عنها كنشاط عام
٤. مطابقة الصفة للموصوف بالتذكير والتأنيث

٥. مطابقة المفرد والمثنى والجمع (ركّزي على هذي الفئة بشكل خاص):
   - مطابقة الفعل مع الفاعل بالعدد: فاعل مفرد يحتاج فعل مفرد ("الطالب ذهب")، فاعل مثنى يحتاج فعل مثنى ("الطالبان ذهبا")، فاعل جمع يحتاج فعل جمع ("الطلاب ذهبوا")
   - صيغة المثنى: تنتهي بـ (ان) أو (ين) (مثال: "طالبان"/"طالبين"، مو "طالبان" مع فعل مفرد)
   - جمع المذكر السالم: ينتهي بـ (ون) أو (ين) (مثال: "معلمون"/"معلمين")
   - جمع المؤنث السالم: ينتهي بـ (ات) (مثال: "معلمات")
   - بالاعتماد على حالة الإعراب
   - انتبهي: جمع التكسير (مثل "طلاب"، "رجال") ما له نمط ثابت، فلا تفترضي خطأ فيه إلا لو متأكدة تمامًا

قواعد مهمة جدًا:
- لا تصححي شيء صحيح أصلًا — لو غير متأكدة، لا تصححي.
- لا تصححي العامية/اللهجات.
- لا تكتبي تصحيحًا مطابقًا للكلمة الأصلية بالضبط.

بالإضافة لذلك، اكتبي رسالة "overall_feedback" قصيرة ومناسبة لعمر الطفل:
- لو ما فيه أخطاء: احتفلي بإنجازه.
- لو فيه ١-٢ خطأ: رسالة بسيطة ومختصرة، مو مبالغ فيها.
- لو فيه ٣ أخطاء أو أكثر: طبّعي إن التعلم بالتجربة طبيعي، وشجّعيه، بدون أي كلمة سلبية أو محبطة.

أمثلة على السلوك الصحيح:

مثال ١ (خطأ مطابقة عدد):
المدخل: "الطالبان ذهب الى المدرسة"
المخرج: {"overall_feedback": "شبه ممتاز! بس فيه تصحيح بسيط:", "errors": [{"wrong": "ذهب", "correct": "ذهبا", "explanation": "الفاعل مثنى (الطالبان) فيحتاج فعل مثنى ينتهي بألف"}]}

مثال ٢ (جملة صحيحة تمامًا):
المدخل: "ذهبت إلى المكتبة واستعرت كتابًا جميلًا"
المخرج: {"overall_feedback": "ممتاز! جملتك صحيحة بالكامل! 🎉", "errors": []}

مثال ٣ (اسم علم غير مألوف، لا تخترعي له تصحيحًا وهميًا):
المدخل: "لعبت هياء مع أخيها"
المخرج: {"overall_feedback": "ممتاز! جملتك صحيحة بالكامل! 🎉", "errors": []}

أرجعي فقط JSON بهذا الشكل، بدون أي نص إضافي قبله أو بعده:
{"overall_feedback": "...", "errors": [{"wrong": "...", "correct": "...", "explanation": "..."}]}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ]
    )

    raw = response.choices[0].message.content

    try:
        data = json.loads(raw)
        errors = data["errors"]
        feedback = data.get("overall_feedback", "")
    except (json.JSONDecodeError, KeyError):
        print("تحذير: رد غير صحيح:", raw)
        return "", []

    errors = [e for e in errors if e["wrong"].strip() != e["correct"].strip()]

    return feedback, errors

Spell Checker class

In [13]:
from google.colab import userdata
from spell_checker import SpellChecker

checker = SpellChecker(api_key=userdata.get('GROQ_API_KEY'))
feedback, errors = checker.correct("اليوم رحت الى المدرسه وتعلمت اشياء حلوه")

print(feedback)
for e in errors:
    print(f"{e['wrong']} ← {e['correct']} ({e['explanation']})")

التعلم بالتجربة طبيعي، استمرّ في المحاولة وستتحسّن أكثر!
الى ← إلى (كلمة "إلى" تحتاج همزة على الألف)
المدرسه ← المدرسة (كلمة "المدرسة" تنتهي بتاء مربوطة)
اشياء ← أشياء (كلمة "أشياء" تحتاج همزة على الألف)
حلوه ← حلوة (كلمة "حلوة" تنتهي بتاء مربوطة)
